In [1]:
from pynq import Overlay


In [2]:
ol = Overlay ("design_2.bit")
print("Overlay loaded!")

Overlay loaded!


In [3]:
print(ol.ip_dict)

{'axi_gpio_0': {'type': 'xilinx.com:ip:axi_gpio:2.0', 'mem_id': 'S_AXI', 'memtype': 'REGISTER', 'gpio': {}, 'interrupts': {}, 'parameters': {'C_FAMILY': 'zynq', 'C_S_AXI_ADDR_WIDTH': '9', 'C_S_AXI_DATA_WIDTH': '32', 'C_GPIO_WIDTH': '1', 'C_GPIO2_WIDTH': '32', 'C_ALL_INPUTS': '0', 'C_ALL_INPUTS_2': '0', 'C_ALL_OUTPUTS': '1', 'C_ALL_OUTPUTS_2': '0', 'C_INTERRUPT_PRESENT': '0', 'C_DOUT_DEFAULT': '0x00000000', 'C_TRI_DEFAULT': '0xFFFFFFFF', 'C_IS_DUAL': '0', 'C_DOUT_DEFAULT_2': '0x00000000', 'C_TRI_DEFAULT_2': '0xFFFFFFFF', 'Component_Name': 'design_1_axi_gpio_0_0', 'USE_BOARD_FLOW': 'false', 'GPIO_BOARD_INTERFACE': 'Custom', 'GPIO2_BOARD_INTERFACE': 'Custom', 'EDK_IPTYPE': 'PERIPHERAL', 'C_BASEADDR': '0x41200000', 'C_HIGHADDR': '0x4120FFFF', 'DATA_WIDTH': '32', 'PROTOCOL': 'AXI4LITE', 'FREQ_HZ': '50000000', 'ID_WIDTH': '0', 'ADDR_WIDTH': '9', 'AWUSER_WIDTH': '0', 'ARUSER_WIDTH': '0', 'WUSER_WIDTH': '0', 'RUSER_WIDTH': '0', 'BUSER_WIDTH': '0', 'READ_WRITE_MODE': 'READ_WRITE', 'HAS_BURST': 

In [4]:
print(ol.mem_dict)

{'axi_bram_ctrl_0': {'fullpath': 'axi_bram_ctrl_0', 'type': 'DDR4', 'bdtype': None, 'state': None, 'addr_range': 8192, 'phys_addr': 1073741824, 'mem_id': 'S_AXI', 'memtype': 'MEMORY', 'gpio': {}, 'interrupts': {}, 'parameters': {'C_BRAM_INST_MODE': 'EXTERNAL', 'C_MEMORY_DEPTH': '2048', 'C_BRAM_ADDR_WIDTH': '11', 'C_S_AXI_ADDR_WIDTH': '13', 'C_S_AXI_DATA_WIDTH': '32', 'C_S_AXI_ID_WIDTH': '1', 'C_S_AXI_PROTOCOL': 'AXI4', 'C_S_AXI_SUPPORTS_NARROW_BURST': '0', 'C_SINGLE_PORT_BRAM': '1', 'C_FAMILY': 'zynq', 'C_READ_LATENCY': '1', 'C_RD_CMD_OPTIMIZATION': '0', 'C_S_AXI_CTRL_ADDR_WIDTH': '32', 'C_S_AXI_CTRL_DATA_WIDTH': '32', 'C_ECC': '0', 'C_ECC_TYPE': '0', 'C_FAULT_INJECT': '0', 'C_ECC_ONOFF_RESET_VALUE': '0', 'DATA_WIDTH': '32', 'ID_WIDTH': '0', 'PROTOCOL': 'AXI4', 'SUPPORTS_NARROW_BURST': '0', 'SINGLE_PORT_BRAM': '1', 'ECC_TYPE': '0', 'USE_ECC': '0', 'FAULT_INJECT': '0', 'ECC_ONOFF_RESET_VALUE': '0', 'Component_Name': 'design_1_axi_bram_ctrl_0_0', 'BMG_INSTANCE': 'EXTERNAL', 'MEM_DEPTH': 

In [5]:
print("IP devices:")
for name in ol.ip_dict.keys():
    print(name)

print("\nMemory devices:")
for name in ol.mem_dict.keys():
    print(name)

IP devices:
axi_gpio_0
processing_system7_0

Memory devices:
axi_bram_ctrl_0
axi_bram_ctrl_1
PSDDR


In [6]:
from pynq import MMIO

INSTR_BRAM_BASE = ol.mem_dict["axi_bram_ctrl_0"]["phys_addr"]
INSTR_BRAM_SIZE = ol.mem_dict["axi_bram_ctrl_0"]["addr_range"]

DATA_BRAM_BASE = ol.mem_dict["axi_bram_ctrl_1"]["phys_addr"]
DATA_BRAM_SIZE = ol.mem_dict["axi_bram_ctrl_1"]["addr_range"]

GPIO_BASE = ol.ip_dict["axi_gpio_0"]["phys_addr"]
GPIO_SIZE = ol.ip_dict["axi_gpio_0"]["addr_range"]

instr_bram = MMIO(INSTR_BRAM_BASE, INSTR_BRAM_SIZE)
data_bram = MMIO(DATA_BRAM_BASE, DATA_BRAM_SIZE)
gpio_mmio = MMIO(GPIO_BASE, GPIO_SIZE)

print(f"Instruction BRAM: 0x{INSTR_BRAM_BASE:08X}")
print(f"Data BRAM:        0x{DATA_BRAM_BASE:08X}")
print(f"GPIO:             0x{GPIO_BASE:08X}")

Instruction BRAM: 0x40000000
Data BRAM:        0x42000000
GPIO:             0x41200000


In [7]:
GPIO_DATA_OFFSET = 0x00

# GPIO=0，Puts the RISC-V into reset after the NOT operation.
gpio_mmio.write(GPIO_DATA_OFFSET, 0x0)

gpio_data = gpio_mmio.read(GPIO_DATA_OFFSET) & 0x1

print(f"GPIO DATA = {gpio_data}")
print("RISC-V is held in reset.")

GPIO DATA = 0
RISC-V is held in reset.


In [8]:
TEST_OFFSET = 0x00
TEST_VALUE = 0x12345678

data_bram.write(TEST_OFFSET, TEST_VALUE)
read_back = data_bram.read(TEST_OFFSET)

print(f"Written:   0x{TEST_VALUE:08X}")
print(f"Read back: 0x{read_back:08X}")

assert read_back == TEST_VALUE, "Data BRAM test failed."
print("Data BRAM test passed.")

Written:   0x12345678
Read back: 0x12345678
Data BRAM test passed.


In [9]:
TEST_OFFSET = 0x00
RISC_V_NOP = 0x00000013  # addi x0, x0, 0

instr_bram.write(TEST_OFFSET, RISC_V_NOP)
read_back = instr_bram.read(TEST_OFFSET)

print(f"Written:   0x{RISC_V_NOP:08X}")
print(f"Read back: 0x{read_back:08X}")

assert read_back == RISC_V_NOP, "Instruction BRAM test failed."
print("Instruction BRAM test passed.")

Written:   0x00000013
Read back: 0x00000013
Instruction BRAM test passed.


In [10]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles:")
for filename in os.listdir("."):
    print(" ", filename)

Current directory:
/home/xilinx/jupyter_notebooks/project_2

Files:
  sort32_controlled_32.hex
  sort_benchmark_raw.csv
  arithmetic_31.hex
  test_sort.hex
  sort32.hex
  loaded.xclbin
  heap_sort.hex
  sort_benchmark_summary.csv
  sort32_fixed.hex
  riscv_design_2_complete_benchmark.ipynb
  design_1.hwh
  sort_benchmark_comparison.csv
  design_2.bit
  final.ipynb
  design_2.hwh
  heap_sort_v2.hex
  .ipynb_checkpoints
  design_1.bit
  benchmark_figures
  arithmetic_32.hex
  Untitled.ipynb


In [11]:
HEX_FILE = "test_sort.hex"

instructions = []

with open(HEX_FILE, "r") as file:
    for line_number, raw_line in enumerate(file, start=1):
        line = raw_line.strip()

        if not line:
            continue

        # Remove common comments
        line = line.split("#", 1)[0]
        line = line.split("//", 1)[0]
        line = line.strip()

        if not line:
            continue

       # If a line starts with @address, temporarily skip that address tag.
        if line.startswith("@"):
            continue

        try:
            instruction = int(line, 16)
        except ValueError as error:
            raise ValueError(
                f"Line {line_number} is not valid hexadecimal: {line}"
            ) from error

        if instruction > 0xFFFFFFFF:
            raise ValueError(
                f"Line {line_number} exceeds 32 bits: {line}"
            )

        instructions.append(instruction)

print(f"Loaded {len(instructions)} instructions.")

for index, instruction in enumerate(instructions[:10]):
    print(f"{index:04d}: 0x{instruction:08X}")

Loaded 42 instructions.
0000: 0x00500093
0001: 0x00A00113
0002: 0x002081B3
0003: 0x00302023
0004: 0x40110233
0005: 0x00402223
0006: 0x0020F2B3
0007: 0x00502423
0008: 0x0020E333
0009: 0x00602623


In [12]:
# Write and verify Instruction BRAM.
MAX_INSTRUCTIONS = INSTR_BRAM_SIZE // 4

print(f"Program instructions: {len(instructions)}")
print(f"Instruction BRAM capacity: {MAX_INSTRUCTIONS}")

# 1. Check whether the program fits into the Instruction BRAM.
if len(instructions) > MAX_INSTRUCTIONS:
    raise ValueError(
        f"Program is too large: {len(instructions)} instructions, "
        f"capacity is {MAX_INSTRUCTIONS}."
    )

print("Program fits in Instruction BRAM.")

# 2. Write all instructions to Instruction BRAM.
for index, instruction in enumerate(instructions):
    byte_offset = index * 4
    instr_bram.write(byte_offset, instruction)

print(f"Wrote {len(instructions)} instructions to Instruction BRAM.")

# 3. Read back from Instruction BRAM and compare
mismatches = []

for index, expected in enumerate(instructions):
    byte_offset = index * 4
    actual = instr_bram.read(byte_offset)

    if actual != expected:
        mismatches.append({
            "index": index,
            "offset": byte_offset,
            "expected": expected,
            "actual": actual
        })

# 4. output verify result
if not mismatches:
    print("Instruction BRAM verification passed.")
else:
    print(f"Found {len(mismatches)} mismatches.")

    for item in mismatches[:10]:
        print(
            f"index={item['index']}, "
            f"offset=0x{item['offset']:04X}, "
            f"expected=0x{item['expected']:08X}, "
            f"actual=0x{item['actual']:08X}"
        )

    raise RuntimeError("Instruction BRAM verification failed.")

Program instructions: 42
Instruction BRAM capacity: 2048
Program fits in Instruction BRAM.
Wrote 42 instructions to Instruction BRAM.
Instruction BRAM verification passed.


In [13]:
# check test enviornment
required_variables = [
    "instr_bram",
    "data_bram",
    "gpio_mmio",
    "INSTR_BRAM_SIZE",
    "DATA_BRAM_SIZE",
    "instructions"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Please run the previous setup cells first. "
        f"Missing variables: {missing_variables}"
    )

print("All required objects are ready.")
print(f"Program instructions: {len(instructions)}")
print(f"Instruction BRAM size: {INSTR_BRAM_SIZE} bytes")
print(f"Data BRAM size: {DATA_BRAM_SIZE} bytes")

All required objects are ready.
Program instructions: 42
Instruction BRAM size: 8192 bytes
Data BRAM size: 8192 bytes


In [14]:
#cpu keep reset

import time

GPIO_DATA_OFFSET = 0x00

# 0x00～0x3C：16 function test result
RESULT_OFFSETS = list(range(0x00, 0x40, 4))

# DONE store at 0x0080
DONE_OFFSET = 0x80
EXPECTED_DONE = 0x00000001

# gpio_run = 0
# through not gate ：cpu_reset = 1
gpio_mmio.write(GPIO_DATA_OFFSET, 0x00000000)
time.sleep(0.02)

print("gpio_run = 0")
print("RISC-V CPU is held in reset.")

# clear result add
for offset in RESULT_OFFSETS:
    data_bram.write(offset, 0x00000000)

# clear DONE
data_bram.write(DONE_OFFSET, 0x00000000)

# verify clear
clear_failed = False

for offset in RESULT_OFFSETS:
    actual = data_bram.read(offset)

    if actual != 0:
        print(
            f"Clear failed at offset 0x{offset:02X}: "
            f"0x{actual:08X}"
        )
        clear_failed = True

done_before = data_bram.read(DONE_OFFSET)

if done_before != 0:
    print(
        f"Clear failed at DONE offset 0x{DONE_OFFSET:02X}: "
        f"0x{done_before:08X}"
    )
    clear_failed = True

if clear_failed:
    raise RuntimeError("Could not clear the CPU test result area.")

print("Data BRAM test result area cleared.")
print(f"DONE offset = 0x{DONE_OFFSET:02X}")

gpio_run = 0
RISC-V CPU is held in reset.
Data BRAM test result area cleared.
DONE offset = 0x80


In [15]:
##run FPGA RISC-V CPU

gpio_mmio.write(GPIO_DATA_OFFSET, 0x00000001)

print("gpio_run = 1")
print("CPU reset released.")
print("The FPGA RISC-V CPU is now executing the function test.")

gpio_run = 1
CPU reset released.
The FPGA RISC-V CPU is now executing the function test.


In [16]:
#Polling the DONE flag for the full-functionality test
import time

TIMEOUT_SECONDS = 2.0
POLL_INTERVAL_SECONDS = 0.01

start_time = time.monotonic()
finished = False
last_done = 0

while time.monotonic() - start_time < TIMEOUT_SECONDS:
    last_done = data_bram.read(DONE_OFFSET)

    if last_done == EXPECTED_DONE:
        finished = True
        break

    time.sleep(POLL_INTERVAL_SECONDS)

if finished:
    elapsed = time.monotonic() - start_time

    print(
        f"DONE detected: 0x{last_done:08X}, "
        f"elapsed={elapsed:.4f} seconds"
    )
else:
    print("CPU function test timed out.")
    print(f"Expected DONE = 0x{EXPECTED_DONE:08X}")
    print(f"Actual DONE   = 0x{last_done:08X}")

DONE detected: 0x00000001, elapsed=0.0030 seconds


In [17]:
# Cell 12: Stop the CPU and check all CPU functions one by one

import time

# Put CPU back into reset
gpio_mmio.write(GPIO_DATA_OFFSET, 0x00000000)
time.sleep(0.02)

print("CPU returned to reset.")

expected_results = {
    "ADD / forwarding":   (0x00, 0x0000000F),
    "SUB":                (0x04, 0x00000005),
    "AND":                (0x08, 0x00000000),
    "OR":                 (0x0C, 0x0000000F),
    "XOR":                (0x10, 0x0000000F),
    "SLL":                (0x14, 0x000000A0),
    "SRL":                (0x18, 0x00000005),
    "SRA":                (0x1C, 0xFFFFFFFF),
    "SLT signed":         (0x20, 0x00000001),
    "SLTU unsigned":      (0x24, 0x00000000),
    "LW":                 (0x28, 0x0000000F),
    "LW-use stall":       (0x2C, 0x00000010),
    "BEQ / flush":        (0x30, 0x00000001),
    "LUI":                (0x34, 0x12345000),
    "JAL jump":           (0x38, 0x00000001),
    "JAL return address": (0x3C, 0x00000088),
    "DONE":               (0x80, 0x00000001),
}

all_passed = True
failed_tests = []

print()
print("=" * 100)
print("FPGA RISC-V CPU FUNCTION TEST")
print("=" * 100)

for test_name, (offset, expected) in expected_results.items():

    actual = data_bram.read(offset) & 0xFFFFFFFF
    passed = actual == expected

    status = "PASS" if passed else "FAIL"

    print(
        f"{test_name:<22} "
        f"offset=0x{offset:02X}  "
        f"expected=0x{expected:08X}  "
        f"actual=0x{actual:08X}  "
        f"{status}"
    )

    if not passed:
        all_passed = False
        failed_tests.append(
            {
                "name": test_name,
                "offset": offset,
                "expected": expected,
                "actual": actual
            }
        )

print("=" * 100)

# Check whether CPU actually reached the end
done_value = data_bram.read(0x80) & 0xFFFFFFFF
finished = (done_value == 0x00000001)

if all_passed and finished:
    print("FINAL RESULT: PASS")
    print("All tested CPU functions are working correctly.")

else:
    print("FINAL RESULT: FAIL")

    if not finished:
        print()
        print("WARNING:")
        print("CPU did not reach the DONE store.")
        print("Some later FAIL results may be caused by an earlier control-flow error.")

    if failed_tests:
        print()
        print("Failed tests:")

        for test in failed_tests:
            print(
                f"  {test['name']:<22} "
                f"expected=0x{test['expected']:08X} "
                f"actual=0x{test['actual']:08X}"
            )

print("=" * 100)

CPU returned to reset.

FPGA RISC-V CPU FUNCTION TEST
ADD / forwarding       offset=0x00  expected=0x0000000F  actual=0x0000000F  PASS
SUB                    offset=0x04  expected=0x00000005  actual=0x00000005  PASS
AND                    offset=0x08  expected=0x00000000  actual=0x00000000  PASS
OR                     offset=0x0C  expected=0x0000000F  actual=0x0000000F  PASS
XOR                    offset=0x10  expected=0x0000000F  actual=0x0000000F  PASS
SLL                    offset=0x14  expected=0x000000A0  actual=0x000000A0  PASS
SRL                    offset=0x18  expected=0x00000005  actual=0x00000005  PASS
SRA                    offset=0x1C  expected=0xFFFFFFFF  actual=0xFFFFFFFF  PASS
SLT signed             offset=0x20  expected=0x00000001  actual=0x00000001  PASS
SLTU unsigned          offset=0x24  expected=0x00000000  actual=0x00000000  PASS
LW                     offset=0x28  expected=0x0000000F  actual=0x0000000F  PASS
LW-use stall           offset=0x2C  expected=0x00000010

In [18]:
import time

GPIO_DATA_OFFSET = 0x00

#CPU logic add
CPU_DATA_BASE_ADDRESS = 0x1000
CPU_STATUS_ADDRESS = 0x2000


# PYNQ access Data BRAM offset

RESULT_OFFSET = 0x0000

STATUS_OFFSET = (
    CPU_STATUS_ADDRESS - CPU_DATA_BASE_ADDRESS
)

MAGIC_NUMBER = 0xCAFEBABE

print(f"Data BRAM size      = 0x{DATA_BRAM_SIZE:04X} bytes")
print(f"CPU data address    = 0x{CPU_DATA_BASE_ADDRESS:04X}")
print(f"CPU status address  = 0x{CPU_STATUS_ADDRESS:04X}")
print(f"PYNQ result offset  = 0x{RESULT_OFFSET:04X}")
print(f"PYNQ status offset  = 0x{STATUS_OFFSET:04X}")

if STATUS_OFFSET + 4 > DATA_BRAM_SIZE:
    raise RuntimeError(
        "The translated status offset is outside Data BRAM."
    )

print("Status offset is inside Data BRAM.")
# ============================================================
# minimal test programm
# ============================================================

# logic：
#
# x3 = 0x1000
# x1 = 15
# DMEM[0x1000] = 15
#
# x4 = 0x2000
# x2 = 0xCAFEBABE
# DMEM[0x2000] = 0xCAFEBABE
#
# loop

minimal_program = [
    0x000011B7,  # lui  x3, 0x1
                 # x3 = 0x00001000

    0x00F00093,  # addi x1, x0, 15
                 # x1 = 15

    0x0011A023,  # sw   x1, 0(x3)
                 # DMEM[0x1000] = 15

    0x00002237,  # lui  x4, 0x2
                 # x4 = 0x00002000

    0xCAFEC137,  # lui  x2, 0xCAFEC
                 # x2 = 0xCAFEC000

    0xABE10113,  # addi x2, x2, -1346
                 # x2 = 0xCAFEBABE

    0x00222023,  # sw   x2, 0(x4)
                 # DMEM[0x2000] = 0xCAFEBABE

    0x0000006F,  # jal  x0, 0
                 # loop

]
#keeping cpu reset

gpio_mmio.write(GPIO_DATA_OFFSET, 0)
time.sleep(0.02)

print(
    "GPIO before loading =",
    gpio_mmio.read(GPIO_DATA_OFFSET) & 0x1
)


#write the program
    
for index, instruction in enumerate(minimal_program):
    instr_bram.write(index * 4, instruction)


#3. Read back and verify the Instruction BRAM.
for index, expected in enumerate(minimal_program):
    offset = index * 4
    actual = instr_bram.read(offset) & 0xFFFFFFFF

    print(
        f"IMEM offset=0x{offset:04X}, "
        f"expected=0x{expected:08X}, "
        f"actual=0x{actual:08X}"
    )

    if actual != expected:
        raise RuntimeError(
            f"Instruction BRAM mismatch at offset 0x{offset:X}"
        )


#Clear the results and status in Data BRAM.

data_bram.write(RESULT_OFFSET, 0x00000000)
data_bram.write(STATUS_OFFSET, 0x00000000)

result_before = data_bram.read(RESULT_OFFSET) & 0xFFFFFFFF
status_before = data_bram.read(STATUS_OFFSET) & 0xFFFFFFFF

print()
print(f"Before run result = 0x{result_before:08X}")
print(f"Before run status = 0x{status_before:08X}")

if result_before != 0 or status_before != 0:
    raise RuntimeError(
        "Could not clear the result or status location."
    )

# 5. Release CPU reset

gpio_mmio.write(GPIO_DATA_OFFSET, 1)

print()
print("GPIO run = 1")
print("CPU reset released.")

time.sleep(0.20)


# 6. Reset the CPU again
gpio_mmio.write(GPIO_DATA_OFFSET, 0)
time.sleep(0.02)

print("GPIO run = 0")
print("CPU returned to reset.")


# 7. Read results and status flag

result = data_bram.read(RESULT_OFFSET) & 0xFFFFFFFF
status = data_bram.read(STATUS_OFFSET) & 0xFFFFFFFF

print()
print(f"Result at 0x{RESULT_OFFSET:04X} = 0x{result:08X} ({result})")
print(f"Status at 0x{STATUS_OFFSET:04X} = 0x{status:08X}")


# 8. Evaluate test results

if result == 15 and status == MAGIC_NUMBER:
    print()
    print("PASS: CPU accessed the correct Data BRAM locations.")
    print("PASS: Status magic number was written to 0x2000.")
else:
    print()
    print("FAIL: CPU did not produce the expected values.")

    if result != 15:
        print(
            f"Result mismatch: "
            f"expected 15, actual {result}"
        )

    if status != MAGIC_NUMBER:
        print(
            f"Status mismatch: "
            f"expected 0x{MAGIC_NUMBER:08X}, "
            f"actual 0x{status:08X}"
        )

Data BRAM size      = 0x2000 bytes
CPU data address    = 0x1000
CPU status address  = 0x2000
PYNQ result offset  = 0x0000
PYNQ status offset  = 0x1000
Status offset is inside Data BRAM.
GPIO before loading = 0
IMEM offset=0x0000, expected=0x000011B7, actual=0x000011B7
IMEM offset=0x0004, expected=0x00F00093, actual=0x00F00093
IMEM offset=0x0008, expected=0x0011A023, actual=0x0011A023
IMEM offset=0x000C, expected=0x00002237, actual=0x00002237
IMEM offset=0x0010, expected=0xCAFEC137, actual=0xCAFEC137
IMEM offset=0x0014, expected=0xABE10113, actual=0xABE10113
IMEM offset=0x0018, expected=0x00222023, actual=0x00222023
IMEM offset=0x001C, expected=0x0000006F, actual=0x0000006F

Before run result = 0x00000000
Before run status = 0x00000000

GPIO run = 1
CPU reset released.
GPIO run = 0
CPU returned to reset.

Result at 0x0000 = 0xCAFEBABE (3405691582)
Status at 0x1000 = 0x0000000F

FAIL: CPU did not produce the expected values.
Result mismatch: expected 15, actual 3405691582
Status mismatch

In [19]:
# SORT Cell 1: Sort Test Configuration and Physical Address Mapping

import time

GPIO_DATA_OFFSET = 0x00

SORT_HEX_FILE = "sort32_fixed.hex"

# Logical addresses used by the CPU program

# The 32 data items to be sorted start at CPU address 0x1000.
CPU_ARRAY_ADDRESS = 0x1000

# After sorting is complete, the CPU writes the magic number to address 0x2000.
CPU_STATUS_ADDRESS = 0x2000

ARRAY_COUNT = 32
ARRAY_SIZE_BYTES = ARRAY_COUNT * 4

# Actual Python MMIO offset determined via minimal program testing

# CPU address 0x1000 corresponds to Data BRAM offset 0x1000
ARRAY_OFFSET = 0x1000

# CPU address 0x2000 corresponds to Data BRAM offset 0x0000.
STATUS_OFFSET = 0x0000

# Sorting completion flag
MAGIC_NUMBER = 0xCAFEBABE

# Polling Configuration
TIMEOUT_SECONDS = 2.0
POLL_INTERVAL_SECONDS = 0.01

# Check Data BRAM range

if ARRAY_OFFSET + ARRAY_SIZE_BYTES > DATA_BRAM_SIZE:
    raise RuntimeError(
        "The sorting array is outside the Data BRAM range."
    )

if STATUS_OFFSET + 4 > DATA_BRAM_SIZE:
    raise RuntimeError(
        "The status location is outside the Data BRAM range."
    )

#print out the add

print("Sorting test configured.")

print()
print("CPU logical address layout:")
print(f"Array count          : {ARRAY_COUNT}")
print(
    f"Array address range  : "
    f"0x{CPU_ARRAY_ADDRESS:04X} - "
    f"0x{CPU_ARRAY_ADDRESS + ARRAY_SIZE_BYTES - 4:04X}"
)
print(f"Status address       : 0x{CPU_STATUS_ADDRESS:04X}")
print(f"Expected magic number: 0x{MAGIC_NUMBER:08X}")

print()
print("Python Data BRAM MMIO layout:")
print(
    f"Array offset range   : "
    f"0x{ARRAY_OFFSET:04X} - "
    f"0x{ARRAY_OFFSET + ARRAY_SIZE_BYTES - 4:04X}"
)
print(f"Status offset        : 0x{STATUS_OFFSET:04X}")
print(f"Data BRAM size       : 0x{DATA_BRAM_SIZE:04X} bytes")

Sorting test configured.

CPU logical address layout:
Array count          : 32
Array address range  : 0x1000 - 0x107C
Status address       : 0x2000
Expected magic number: 0xCAFEBABE

Python Data BRAM MMIO layout:
Array offset range   : 0x1000 - 0x107C
Status offset        : 0x0000
Data BRAM size       : 0x2000 bytes


In [20]:
# SORT Cell 2: Prepare 32 random test data samples for sorting
import random

input_values = [
    random.randint(-1000, 1000)
    for _ in range(ARRAY_COUNT)
]

if len(input_values) != ARRAY_COUNT:
    raise RuntimeError(
        f"Expected {ARRAY_COUNT} input values, "
        f"but received {len(input_values)}."
    )

golden_result = sorted(input_values)

print("Random input values:")
print(input_values)

print()
print("Python golden result:")
print(golden_result)

print()
print(f"Input count: {len(input_values)}")
print(f"Minimum value: {min(input_values)}")
print(f"Maximum value: {max(input_values)}")

Random input values:
[-501, 842, -742, 757, 816, -524, -618, -48, 761, -11, -682, 691, -220, 340, -586, 375, -916, -539, -956, -921, -833, 858, 561, -995, -893, 51, -4, -118, -924, -101, -163, -964]

Python golden result:
[-995, -964, -956, -924, -921, -916, -893, -833, -742, -682, -618, -586, -539, -524, -501, -220, -163, -118, -101, -48, -11, -4, 51, 340, 375, 561, 691, 757, 761, 816, 842, 858]

Input count: 32
Minimum value: -995
Maximum value: 858


In [21]:
# SORT Cell 3: 32-bit signed/unsigned conversion

def to_unsigned32(value):
    return int(value) & 0xFFFFFFFF


def to_signed32(value):
    value = int(value) & 0xFFFFFFFF

    if value & 0x80000000:
        return value - 0x100000000

    return value


print("Conversion functions ready.")

Conversion functions ready.


In [22]:
# SORT Cell 4: Read sorting program

sort_instructions = []

with open(SORT_HEX_FILE, "r") as file:
    for raw_line in file:
        line = raw_line.split("#", 1)[0]
        line = line.split("//", 1)[0]
        line = line.strip()

        if not line or line.startswith("@"):
            continue

        sort_instructions.append(int(line, 16))

if not sort_instructions:
    raise RuntimeError("No sorting instructions were loaded.")

print(f"Loaded {len(sort_instructions)} sorting instructions.")

for index, instruction in enumerate(sort_instructions):
    print(
        f"{index:02d}: "
        f"offset=0x{index * 4:04X}, "
        f"instruction=0x{instruction:08X}"
    )

Loaded 31 sorting instructions.
00: offset=0x0000, instruction=0x01F00113
01: offset=0x0004, instruction=0x00000193
02: offset=0x0008, instruction=0x06218063
03: offset=0x000C, instruction=0x40310233
04: offset=0x0010, instruction=0x00000293
05: offset=0x0014, instruction=0x00001337
06: offset=0x0018, instruction=0x00000013
07: offset=0x001C, instruction=0x00000013
08: offset=0x0020, instruction=0x00032383
09: offset=0x0024, instruction=0x00432403
10: offset=0x0028, instruction=0x00000013
11: offset=0x002C, instruction=0x00000013
12: offset=0x0030, instruction=0x007424B3
13: offset=0x0034, instruction=0x00000013
14: offset=0x0038, instruction=0x00048863
15: offset=0x003C, instruction=0x00832023
16: offset=0x0040, instruction=0x00732223
17: offset=0x0044, instruction=0x00000013
18: offset=0x0048, instruction=0x00430313
19: offset=0x004C, instruction=0x00000013
20: offset=0x0050, instruction=0x00000013
21: offset=0x0054, instruction=0x00128293
22: offset=0x0058, instruction=0x00428463
23

In [23]:
# SORT Cell 5: Hold CPU in reset
gpio_mmio.write(GPIO_DATA_OFFSET, 0)
time.sleep(0.02)

gpio_state = gpio_mmio.read(GPIO_DATA_OFFSET) & 0x1

print(f"GPIO run bit = {gpio_state}")
print("CPU is held in reset.")

GPIO run bit = 0
CPU is held in reset.


In [24]:
# SORT Cell 6: Write and verify Instruction BRAM

if len(sort_instructions) > INSTR_BRAM_SIZE // 4:
    raise RuntimeError("Sorting program is too large.")

for index, instruction in enumerate(sort_instructions):
    instr_bram.write(index * 4, instruction)

for index, expected in enumerate(sort_instructions):
    actual = instr_bram.read(index * 4) & 0xFFFFFFFF

    if actual != expected:
        raise RuntimeError(
            f"Instruction mismatch at index {index}: "
            f"expected 0x{expected:08X}, "
            f"actual 0x{actual:08X}"
        )

print(
    f"Sorting program written and verified: "
    f"{len(sort_instructions)} instructions."
)

Sorting program written and verified: 31 instructions.


In [25]:
# SORT Cell 7: Write to input array and clear status
# CPU keeping reset
gpio_mmio.write(GPIO_DATA_OFFSET, 0)
time.sleep(0.02)

# clear old magic number
data_bram.write(STATUS_OFFSET, 0)

# 32 random number
for index, value in enumerate(input_values):
    offset = ARRAY_OFFSET + index * 4
    data_bram.write(offset, to_unsigned32(value))

# read back array
read_back = [
    to_signed32(
        data_bram.read(ARRAY_OFFSET + index * 4)
    )
    for index in range(ARRAY_COUNT)
]

# read back status
status_before = data_bram.read(STATUS_OFFSET) & 0xFFFFFFFF

print("Data BRAM input:")
print(read_back)

print(f"\nStatus before run: 0x{status_before:08X}")

if read_back != input_values:
    raise RuntimeError("Input array write failed.")

if status_before != 0:
    raise RuntimeError("Status was not cleared.")

print("Input array written successfully.")
print("Status cleared successfully.")

Data BRAM input:
[-501, 842, -742, 757, 816, -524, -618, -48, 761, -11, -682, 691, -220, 340, -586, 375, -916, -539, -956, -921, -833, 858, 561, -995, -893, 51, -4, -118, -924, -101, -163, -964]

Status before run: 0x00000000
Input array written successfully.
Status cleared successfully.


In [26]:
# SORT Cell 8: Pre-start verification of Data BRAM

before_run = [
    to_signed32(
        data_bram.read(ARRAY_OFFSET + index * 4)
    )
    for index in range(ARRAY_COUNT)
]

status_before = (
    data_bram.read(STATUS_OFFSET) & 0xFFFFFFFF
)

print("Input array:")
print(input_values)

print("\nData BRAM before CPU starts:")
print(before_run)

print(f"\nStatus before CPU starts: 0x{status_before:08X}")

if before_run != input_values:
    raise RuntimeError(
        "Input array verification failed."
    )

if status_before != 0:
    raise RuntimeError(
        "Status is not zero before CPU starts."
    )

print("Input array verification passed.")
print("Status is clear.")
print("CPU is ready to run.")

Input array:
[-501, 842, -742, 757, 816, -524, -618, -48, 761, -11, -682, 691, -220, 340, -586, 375, -916, -539, -956, -921, -833, 858, 561, -995, -893, 51, -4, -118, -924, -101, -163, -964]

Data BRAM before CPU starts:
[-501, 842, -742, 757, 816, -524, -618, -48, 761, -11, -682, 691, -220, 340, -586, 375, -916, -539, -956, -921, -833, 858, 561, -995, -893, 51, -4, -118, -924, -101, -163, -964]

Status before CPU starts: 0x00000000
Input array verification passed.
Status is clear.
CPU is ready to run.


In [27]:
# SORT Cell 9: Start CPU and wait for magic number
# Release reset and start the CPU.
gpio_mmio.write(GPIO_DATA_OFFSET, 1)

start_time = time.monotonic()
status = 0

while time.monotonic() - start_time < TIMEOUT_SECONDS:
    status = data_bram.read(STATUS_OFFSET) & 0xFFFFFFFF

    if status == MAGIC_NUMBER:
        break

    time.sleep(POLL_INTERVAL_SECONDS)

elapsed = time.monotonic() - start_time

# Reset the CPU after sorting is complete.
gpio_mmio.write(GPIO_DATA_OFFSET, 0)
time.sleep(0.02)

print(f"Status: 0x{status:08X}")
print(f"Elapsed time: {elapsed:.4f} seconds")

if status != MAGIC_NUMBER:
    raise RuntimeError(
        f"Magic number not detected. "
        f"Expected 0x{MAGIC_NUMBER:08X}, "
        f"actual 0x{status:08X}"
    )

print("Magic number detected.")
print("CPU sorting program completed.")

Status: 0xCAFEBABE
Elapsed time: 0.0020 seconds
Magic number detected.
CPU sorting program completed.


In [28]:
# SORT Cell 10: Read and verify sorting results
hardware_result = [
    to_signed32(
        data_bram.read(ARRAY_OFFSET + index * 4)
    )
    for index in range(ARRAY_COUNT)
]

status = data_bram.read(STATUS_OFFSET) & 0xFFFFFFFF

print("Original input:")
print(input_values)

print("\nHardware result:")
print(hardware_result)

print("\nPython golden result:")
print(golden_result)

print(f"\nStatus: 0x{status:08X}")

if status != MAGIC_NUMBER:
    raise RuntimeError("Magic number is incorrect.")

if hardware_result != golden_result:
    raise RuntimeError("Hardware sorting result is incorrect.")

print("\nPASS: Magic number is correct.")
print("PASS: Hardware sorting result is correct.")

Original input:
[-501, 842, -742, 757, 816, -524, -618, -48, 761, -11, -682, 691, -220, 340, -586, 375, -916, -539, -956, -921, -833, 858, 561, -995, -893, 51, -4, -118, -924, -101, -163, -964]

Hardware result:
[-995, -964, -956, -924, -921, -916, -893, -833, -742, -682, -618, -586, -539, -524, -501, -220, -163, -118, -101, -48, -11, -4, 51, 340, 375, 561, 691, 757, 761, 816, 842, 858]

Python golden result:
[-995, -964, -956, -924, -921, -916, -893, -833, -742, -682, -618, -586, -539, -524, -501, -220, -163, -118, -101, -48, -11, -4, 51, 340, 375, 561, 691, 757, 761, 816, 842, 858]

Status: 0xCAFEBABE

PASS: Magic number is correct.
PASS: Hardware sorting result is correct.
